In [0]:
%run "./03-invoice-stream"

In [0]:
from pyspark.sql.functions import *
import shutil

In [0]:
class invoiceStreamTestSuite():
    def __init__(self):
        self.base_data_dir = "/Volumes/streamingdata/dbo/"

    def cleanTests(self):
        print(f"Starting Cleanup...", end = '')
        spark.sql("drop table if exists invoice_line_items")
        dbutils.fs.rm("streamingdata.dbo.invoice_line_items", True)

        dbutils.fs.rm(f"{self.base_data_dir}/checkpoint/invoices", True)
        dbutils.fs.rm(f"{self.base_data_dir}/data/invoices", True)

        dbutils.fs.mkdirs(f"{self.base_data_dir}/data/invoices")
        print("Done")

    def ingestData(self, itr):
        print(f"\tStarting Ingestion...", end='')
        shutil.copy(f"{self.base_data_dir}/raw/invoices_{itr}.json", f"{self.base_data_dir}/data/")
        print("Done\n")
        
    def assertResult(self, expected_count):
        print(f"\tStarting Validation...", end = '')
        actual_count = spark.sql("select count(*) from invoice_line_items").collect()[0][0]
        assert_expected_count == actual_count, f"Test Failed! actual count is {actual_count}"
        print("Done\n")

    def waitForMicroBatch(self, sleep=30):
        import time
        print(f"\tWaiting for {sleep} seconds...", end = '')
        time.sleep(sleep)
        print("Done\n")


    def runTests(self):
        self.cleanTests()
        iStream = invoiceStream()
        streamQuery = iStream.process()

        print("Testing first iteration of invoice stream")
        self.ingestData(1)
        self.waitForMicroBatch()
        self.assertResult(1249)
        print("Validation passed.\n")

        print("Testing first iteration of invoice stream")
        self.ingestData(2)
        self.waitForMicroBatch()
        self.assertResult(2506)
        print("Validation passed.\n")

        print("Testing third iteration of invoice stream")
        self.ingestData(3)
        self.waitForMicroBatch()
        self.assertResult(3990)
        print("Validation passed.\n")

In [0]:
isTs = invoiceStreamTestSuite()
isTs.runTests()